# Débogage DINO — petite zone, mode `tangent`

Avant de regarder les embeddings, ce notebook vérifie **les images que DINOv3 reçoit réellement** en mode
`projection="tangent"`, sur une zone réduite à **8 × 8 tuiles** de `PARENT_LEVEL`. Tout tourne en quelques
secondes, ce qui permet de dérouler les tests un par un.

| Test | Ce qu'on regarde |
|---|---|
| **1** | Image réduite de la zone, et en dessous les 8 × 8 tuiles tangentes dans la même disposition. Chaque tuile porte le contour de sa cellule parente : son contenu doit correspondre au sous-bloc de même position. |
| **1b** | Contrôle chiffré : chaque pixel de tuile comparé à la cellule HEALPix sous lui, et la cellule entièrement contenue dans sa tuile. |
| **1c** | Zoom sur une tuile : grille des patchs 16 px, nord en haut. |
| **2** | `over_sample` : la fenêtre décalée de `(dy, dx)` pixels doit être la tuile de base déplacée vers le **sud / l'est**. |

Les tuiles sont construites avec **les mêmes primitives et les mêmes paramètres** que `GetDINOV3SAT(projection="tangent")`
(`pow2_cover_px`, `tangent_grid_lonlat`, `sample_healpix`, remplissage `fill="mean"`), mais seulement pour les 64 cellules
de la zone. La cellule `[1]` vérifie qu'on obtient bien les mêmes pixels que `tangent_tiles`.


## Paramètres

In [ ]:
SOURCE        = "g4e"        # "g4e" : produits GRID4EARTH  |  "synthetic" : motif connu, sans réseau
PRODUCT_INDEX = 0            # quel produit de G4E_PRODUCTS
LEVEL         = 20           # niveau HEALPix des données
PARENT_LEVEL  = 12           # niveau des tuiles DINO (LEVEL - PARENT_LEVEL >= 4)
GRID          = 8            # la zone fait GRID x GRID tuiles (puissance de 2)
ZONE_LONLAT   = None         # (lon, lat) en degrés pour imposer la zone ; None = zone la plus complète
CACHE         = "cache_g4e"  # réflectances sur disque (None pour désactiver)
STRETCH       = (2, 98)      # percentiles d'affichage, communs à TOUTES les vignettes
ZOOM_TILE     = None         # (row, col) de la tuile à zoomer ; None = centre de la grille


In [ ]:
import sys, os, warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle

sys.path.insert(0, os.path.abspath(".."))      # healpix_analyse depuis le dépôt
sys.path.insert(0, os.path.abspath("."))       # g4e_source à côté du notebook
import healpix_analyse.dino as D

ZONE_LEVEL = PARENT_LEVEL - int(np.log2(GRID))
k = LEVEL - PARENT_LEVEL
assert GRID & (GRID - 1) == 0 and ZONE_LEVEL >= 0 and k >= D.DINOV3_PATCH_LEVELS
print(f"données niveau {LEVEL} (~{D.healpix_gsd_m(LEVEL):.1f} m), tuiles niveau {PARENT_LEVEL} "
      f"(~{D.healpix_gsd_m(PARENT_LEVEL)/1e3:.2f} km), zone niveau {ZONE_LEVEL} "
      f"(~{D.healpix_gsd_m(ZONE_LEVEL)/1e3:.1f} km)")

## Données

In [ ]:
def synthetic_source(level, parent_level, zone_level, lon0=2.35, lat0=48.85):
    # Motif connu, défini dans un repère métrique EST / NORD :
    #   routes N-S tous les 1500 m (blanches), E-O tous les 2300 m (jaunes),
    #   disques rouges, et une grande flèche bleue pointant vers le NORD au centre.
    # Sur une tuile correcte : routes N-S verticales, E-O horizontales, flèche vers le haut.
    zone = int(D.cells_at_lonlat(zone_level, np.array([lon0]), np.array([lat0]))[0])
    g = parent_level - zone_level
    kids = (zone << (2 * g)) + np.arange(4 ** g)
    nb = D.cell_neighbours(parent_level, kids).ravel()
    parents = np.unique(np.concatenate([kids, nb[nb >= 0]]))
    kk = level - parent_level
    ids = ((parents[:, None] << (2 * kk)) + np.arange(4 ** kk)[None]).ravel()
    lon, lat = D.cell_centres_lonlat(level, ids)
    zlon, zlat = D.cell_centres_lonlat(zone_level, np.array([zone]))
    xi, eta = D.lonlat_to_offsets(zlon, zlat, lon[None], lat[None])
    X, Y = xi[0] * D.EARTH_RADIUS_M, eta[0] * D.EARTH_RADIUS_M     # mètres, est / nord
    L = D.healpix_gsd_m(zone_level)
    rgb = np.empty((ids.size, 3), np.float32)
    rgb[:] = (0.08, 0.14, 0.06)                                     # fond « végétation »
    rgb[:, 1] += 0.04 * np.sin(X / 700) * np.cos(Y / 900)
    w = 2.5 * D.healpix_gsd_m(level)
    rgb[np.abs((X + 1500 / 2) % 1500 - 1500 / 2) < w] = (0.35, 0.35, 0.35)
    rgb[np.abs((Y + 2300 / 2) % 2300 - 2300 / 2) < w * 0.7] = (0.30, 0.28, 0.08)
    for cx, cy, r in [(0.20, 0.25, 0.04), (-0.30, 0.10, 0.06), (0.05, -0.30, 0.05), (-0.15, -0.20, 0.03)]:
        rgb[np.hypot(X - cx * L, Y - cy * L) < r * L] = (0.35, 0.06, 0.05)
    a = 0.12 * L                                                    # flèche vers le nord
    shaft = (np.abs(X) < a * 0.25) & (Y > -a) & (Y < a)
    head = (Y >= a) & (Y < 2 * a) & (np.abs(X) < (2 * a - Y) * 0.8)
    rgb[shaft | head] = (0.05, 0.10, 0.40)
    rgb[(np.hypot(X - 0.35 * L, Y + 0.35 * L) < 0.03 * L)] = np.nan  # un trou : teste fill="mean"

    class S: pass
    s = S(); s.level, s.cell_id, s.ellipsoid, s.dates = level, ids, "sphere", ["synthétique"]
    s.rgb = lambda i, **kw: rgb
    return s

if SOURCE == "g4e":
    from g4e_source import ProductSeries, G4E_PRODUCTS
    src = ProductSeries(G4E_PRODUCTS, level=LEVEL)
    rgb_all = src.rgb(PRODUCT_INDEX, cache=CACHE)
else:
    src = synthetic_source(LEVEL, PARENT_LEVEL, ZONE_LEVEL)
    rgb_all = src.rgb(0)

D.set_ellipsoid(getattr(src, "ellipsoid", "sphere"))
ids_all = np.asarray(src.cell_id, dtype=np.int64)
print(f"{ids_all.size:,} cellules, ellipsoïde {D.HEALPIX_ELLIPSOID}, date {src.dates[PRODUCT_INDEX if SOURCE == 'g4e' else 0]}")

## Choix de la zone et sous-ensemble

La zone est une cellule de `ZONE_LEVEL` ; ses `GRID²` enfants sont les tuiles. On garde aussi l'anneau de cellules parentes voisines : une tuile tangente déborde de sa cellule.

In [ ]:
finite = np.isfinite(rgb_all).all(axis=1)
zk = 2 * (LEVEL - ZONE_LEVEL)
if ZONE_LONLAT is not None:
    zone = int(D.cells_at_lonlat(ZONE_LEVEL, np.array([ZONE_LONLAT[0]]), np.array([ZONE_LONLAT[1]]))[0])
else:
    zid, cnt = np.unique(ids_all[finite] >> zk, return_counts=True)
    full = cnt / 4 ** (LEVEL - ZONE_LEVEL)
    # la plus complète ; à égalité, la plus proche du centre de la scène
    zlon, zlat = D.cell_centres_lonlat(ZONE_LEVEL, zid)
    mlon, mlat = np.median(zlon), np.median(zlat)
    order = np.lexsort((np.hypot(zlon - mlon, zlat - mlat), -np.round(full, 3)))
    zone = int(zid[order[0]])

g = PARENT_LEVEL - ZONE_LEVEL
kids = (zone << (2 * g)) + np.arange(GRID * GRID)                      # les 64 tuiles
nb = D.cell_neighbours(PARENT_LEVEL, kids).ravel()
ring = np.unique(np.concatenate([kids, nb[nb >= 0]]))
sel = np.isin(ids_all >> (2 * k), ring)
ids, rgb = ids_all[sel], rgb_all[sel]

# position (row, col) de chaque tuile dans la grille : même convention que nested_to_tiles
x, y = D.nested_to_xy(kids - (zone << (2 * g)))
ROW, COL = GRID - 1 - y, x
zlon, zlat = D.cell_centres_lonlat(ZONE_LEVEL, np.array([zone]))
print(f"zone {zone} au niveau {ZONE_LEVEL}, centre {zlon[0]:.4f}E {zlat[0]:.4f}N")
print(f"{ids.size:,} cellules gardées, couverture de la zone "
      f"{np.isin(ids[np.isfinite(rgb).all(1)] >> zk, [zone]).sum() / 4 ** (LEVEL - ZONE_LEVEL):.1%}")

# étirement d'affichage commun à toutes les vignettes
lo, hi = np.nanpercentile(rgb[np.isin(ids >> zk, [zone])], STRETCH, axis=0)
def show(img_chw_or_hwc, chw=True):
    a = np.transpose(img_chw_or_hwc, (1, 2, 0)) if chw else img_chw_or_hwc
    return np.clip((a - lo) / (hi - lo), 0, 1)

## Construction des tuiles — exactement comme `GetDINOV3SAT(projection="tangent")`

In [ ]:
gsd = D.healpix_gsd_m(LEVEL)
H = W = D.pow2_cover_px(kids, PARENT_LEVEL, gsd)       # même règle que GetDINOV3SAT (tile_px=None)
px = gsd / D.EARTH_RADIUS_M

def tangent_tiles_for(centres, shift=(0.0, 0.0), fill="mean"):
    # Copie de D.tangent_tiles restreinte aux centres demandés (au lieu de tous les parents présents).
    # par paquets de ~2e6 pixels, comme tangent_tiles : le stencil bilinéaire d'une zone
    # entière au niveau 20 demanderait plusieurs Go
    centres = np.asarray(centres)
    clon, clat = D.cell_centres_lonlat(PARENT_LEVEL, centres)
    M, chunk = len(centres), max(1, 2_000_000 // (H * W))
    vals = np.empty((M, H, W, 3), np.float32); valid = np.empty((M, H, W), bool)
    lon = np.empty((M, H, W)); lat = np.empty((M, H, W))
    for a in range(0, M, chunk):
        b = min(a + chunk, M)
        lon[a:b], lat[a:b] = D.tangent_grid_lonlat(clon[a:b], clat[a:b], (H, W), px, shift=shift)
        vals[a:b], valid[a:b] = D.sample_healpix(rgb, ids, LEVEL, lon[a:b], lat[a:b], interpolation="bilinear")
    cov = valid.reshape(len(centres), -1).mean(1)
    if fill == "mean":
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            m = np.nan_to_num(np.nanmean(vals.reshape(len(centres), -1, 3), axis=1))
        vals = np.where(np.isfinite(vals), vals, m[:, None, None, :])
    return np.transpose(vals, (0, 3, 1, 2)).astype(np.float32), valid, cov, lon, lat

tiles, valid, cov, tlon, tlat = tangent_tiles_for(kids)
print(f"{len(kids)} tuiles de {H} x {W} px = {H // 16} x {W // 16} patchs ; couverture min {cov.min():.1%}")

# [1] contrôle : mêmes pixels que D.tangent_tiles (sur une tuile, données restreintes à son parent)
c0 = kids[len(kids) // 2]
n0 = D.cell_neighbours(PARENT_LEVEL, np.array([c0]))[:, 0]
m0 = np.isin(ids >> (2 * k), np.r_[c0, n0[n0 >= 0]])               # le parent et son anneau
rt, rc, *_ = D.tangent_tiles(rgb[m0], ids[m0], LEVEL, PARENT_LEVEL, tile_px=(H, W), gsd_m=gsd, fill="nan")
ref = rt[rc == c0]
mine, _, _, lon0, lat0 = tangent_tiles_for(np.array([c0]), fill="nan")
# on compare à l'intérieur du parent, là où l'anneau suffit à compléter le stencil bilinéaire
inside0 = (D.cells_at_lonlat(LEVEL, lon0[0], lat0[0]) >> (2 * k)) == c0
ok = np.isfinite(ref[0]) & np.isfinite(mine[0]) & inside0[None]
print("[1] identique à D.tangent_tiles :", np.allclose(ref[0][ok], mine[0][ok], atol=1e-6),
      f"({ok.sum() // 3:,} pixels comparés)")

def cell_outline_px(cell, level, centre_lon, centre_lat, Hh, Ww, pxx, step=8):
    # contour d'une cellule HEALPix en coordonnées pixel (col, row) d'une image tangente
    blon, blat = D._boundaries_lonlat(level, np.array([cell]), step=step)
    xi, eta = D.lonlat_to_offsets(np.atleast_1d(centre_lon), np.atleast_1d(centre_lat), blon, blat)
    return np.c_[xi[0] / pxx + (Ww - 1) / 2, (Hh - 1) / 2 - eta[0] / pxx]

## Test 1 — image réduite de la zone, et les 8 × 8 tuiles vues par DINO

* **(a)** bloc NESTED exact de la zone, réduit. C'est la disposition de la grille du dessous : la case `(r, c)` est la cellule parente de la tuile `(r, c)`. Le nord est vers le **coin haut-droit**.
* **(b)** même zone en projection tangente nord en haut, avec les 64 cellules numérotées `r,c`.
* **Grille** : la tuile tangente `(r, c)` telle que DINO la reçoit (nord en haut, après `fill="mean"`). Le contour rouge est la cellule parente : il doit contenir le même paysage que la case `(r, c)` de (a), tourné d'environ 45°, et occuper la même place que dans (b).

In [ ]:
# (a) bloc NESTED de la zone, réduit à ~512 px
mz = (ids >> zk) == zone
blk, _, _, _ = D.nested_to_tiles(rgb[mz], ids[mz], LEVEL, ZONE_LEVEL, fill="nan")
S = blk.shape[-1]; f = max(1, S // 512)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    small = np.nanmean(blk[0].reshape(3, S // f, f, S // f, f), axis=(2, 4))

# (b) vue tangente nord-en-haut de toute la zone
OV = 512
Hz, Wz = D.parent_cover_px(np.array([zone]), ZONE_LEVEL, gsd, margin_px=0)
pz = max(Hz, Wz) * 1.05 * px / OV
olon, olat = D.tangent_grid_lonlat(zlon, zlat, OV, pz)
ov, _ = D.sample_healpix(rgb, ids, LEVEL, olon, olat, interpolation="nearest")

fig = plt.figure(figsize=(16, 24))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 2], hspace=0.08, wspace=0.05)
ax = fig.add_subplot(gs[0, 0]); ax.imshow(show(small)); ax.set_title("(a) bloc NESTED de la zone (nord ↗)")
s8 = small.shape[-1] / GRID
for i in range(1, GRID):
    ax.axhline(i * s8 - .5, c="w", lw=.5, alpha=.6); ax.axvline(i * s8 - .5, c="w", lw=.5, alpha=.6)
for r, c in zip(ROW, COL):
    ax.text((c + .5) * s8, (r + .5) * s8, f"{r},{c}", c="w", fontsize=7, ha="center", va="center")
ax.set_xticks([]); ax.set_yticks([])

ax = fig.add_subplot(gs[0, 1]); ax.imshow(show(np.nan_to_num(ov[0], nan=lo.mean()), chw=False))
ax.set_title("(b) même zone, tangente nord en haut")
for cell, r, c in zip(kids, ROW, COL):
    p = cell_outline_px(cell, PARENT_LEVEL, zlon, zlat, OV, OV, pz)
    ax.add_patch(Polygon(p, fill=False, ec="r", lw=.5, alpha=.7))
    ax.text(*p.mean(0), f"{r},{c}", c="w", fontsize=7, fontweight="bold", ha="center", va="center")
ax.set_xlim(0, OV); ax.set_ylim(OV, 0); ax.set_xticks([]); ax.set_yticks([])

sub = gs[1, :].subgridspec(GRID, GRID, hspace=0.12, wspace=0.04)
clon, clat = D.cell_centres_lonlat(PARENT_LEVEL, kids)
for t, (r, c) in enumerate(zip(ROW, COL)):
    ax = fig.add_subplot(sub[r, c])
    ax.imshow(show(tiles[t]))
    ax.add_patch(Polygon(cell_outline_px(kids[t], PARENT_LEVEL, clon[t], clat[t], H, W, px),
                         fill=False, ec="r", lw=.6))
    ax.set_title(f"{r},{c}  {cov[t]:.0%}", fontsize=6, pad=1)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"zone {zone} (niv. {ZONE_LEVEL}) — {GRID}x{GRID} tuiles {H}x{W} px du niveau {PARENT_LEVEL}", y=.9)
plt.show()

## Test 1b — contrôle chiffré

* **écart** : pour chaque pixel de tuile on lit la cellule HEALPix qui le contient (plus proche voisin) et on compare à la valeur interpolée de la tuile. On attend un écart faible et uniforme. Une tuile retournée ou décalée donnerait un écart de l'ordre de la variance de l'image.
* **contour dans l'image** : fraction du contour de la cellule parente qui tombe dans la tuile. Doit valoir 100 % partout, sinon des cellules n'ont pas de token à elles.

In [ ]:
err, err_flip = np.full(len(kids), np.nan), np.full(len(kids), np.nan)
for t in range(len(kids)):                       # tuile par tuile : reste léger au niveau 20
    near, nval = D.sample_healpix(rgb, ids, LEVEL, tlon[t], tlat[t], interpolation="nearest")
    tv, okp = np.transpose(tiles[t], (1, 2, 0)), valid[t] & nval
    if okp.any():
        err[t] = np.mean(np.abs(tv[okp] - near[okp]))
        err_flip[t] = np.mean(np.abs(tv[::-1][okp] - near[okp]))  # référence : tuile retournée haut/bas
inside = np.array([((p := cell_outline_px(kids[t], PARENT_LEVEL, clon[t], clat[t], H, W, px))[:, 0] >= -.5).all()
                   & (p[:, 0] <= W - .5).all() & (p[:, 1] >= -.5).all() & (p[:, 1] <= H - .5).all()
                   for t in range(len(kids))])

grid_err = np.full((GRID, GRID), np.nan); grid_err[ROW, COL] = err
fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
im = axs[0].imshow(grid_err); plt.colorbar(im, ax=axs[0]); axs[0].set_title("écart moyen tuile / cellule")
g2 = np.zeros((GRID, GRID)); g2[ROW, COL] = inside
axs[1].imshow(g2, vmin=0, vmax=1, cmap="RdYlGn"); axs[1].set_title("cellule entière dans la tuile")
plt.show()
print(f"écart moyen {np.nanmean(err):.4f}  (même tuile retournée haut/bas : {np.nanmean(err_flip):.4f})  "
      f"| dynamique de l'image {np.nanmedian(hi - lo):.4f}")
print(f"cellules entièrement dans leur tuile : {inside.sum()}/{inside.size}")

## Test 1c — zoom sur une tuile

Grille jaune : les patchs 16 × 16 px, un token DINO chacun. Rouge : la cellule parente. Pointillés : la zone remplie par `fill="mean"`. Les tokens dont le centre tombe hors du contour rouge ne sont **pas** réutilisés pour les cellules de la tuile : elles prennent le token le plus proche de leur centre.

In [ ]:
zr, zc = ZOOM_TILE if ZOOM_TILE is not None else (GRID // 2, GRID // 2)
t = int(np.flatnonzero((ROW == zr) & (COL == zc))[0])
fig, axs = plt.subplots(1, 2, figsize=(15, 7.5))
ax = axs[0]; ax.imshow(show(tiles[t]))
for i in range(0, max(H, W) + 1, D.DINOV3_PATCH):
    ax.axhline(i - .5, c="y", lw=.3, alpha=.7); ax.axvline(i - .5, c="y", lw=.3, alpha=.7)
ax.add_patch(Polygon(cell_outline_px(kids[t], PARENT_LEVEL, clon[t], clat[t], H, W, px), fill=False, ec="r", lw=1.2))
if (~valid[t]).any():
    ax.contour(~valid[t], levels=[.5], colors="m", linestyles=":", linewidths=.8)
ax.annotate("N", xy=(W * .95, H * .05), xytext=(W * .95, H * .15), color="w",
            arrowprops=dict(arrowstyle="->", color="w"), ha="center")
ax.set_xlim(-.5, W - .5); ax.set_ylim(H - .5, -.5)
ax.set_title(f"tuile ({zr},{zc}) cellule {kids[t]} — ce que DINO reçoit")

mc = (ids >> (2 * k)) == kids[t]
b, _, _, _ = D.nested_to_tiles(rgb[mc], ids[mc], LEVEL, PARENT_LEVEL, fill="nan")
axs[1].imshow(show(np.nan_to_num(b[0], nan=float(lo.mean()))))
axs[1].set_title("la même cellule en bloc NESTED exact (nord ↗)")
for a in axs[1:]: a.set_xticks([]); a.set_yticks([])
plt.show()

## Test 2 — sens du décalage des fenêtres `over_sample`

`GetDINOV3SAT(over_sample=n)` passe `n²` fenêtres décalées de `(a·16/n, b·16/n)` pixels, puis range leurs tokens aux
indices fins `[a::n, b::n]`. Ce rangement suppose que **la ligne `r` de la fenêtre décalée correspond à la ligne `r + dy` de la
tuile de base**, donc un décalage vers le **sud** pour `dy > 0` et vers l'**est** pour `dx > 0`.

On construit les fenêtres décalées avec `tangent_grid_lonlat(shift=...)`, la fonction appelée par `GetDINOV3SAT`, et on
compare les deux hypothèses.

In [ ]:
step = D.DINOV3_PATCH // 2                      # over_sample=2 : décalage de 8 px
few = np.arange(len(kids))[:: max(1, len(kids) // 8)]
base = tangent_tiles_for(kids[few], fill="nan")[0]

def mismatch(a, b, dr, dc):
    # écart moyen entre b[r, c] et a[r + dr, c + dc]
    A = a[..., max(dr, 0):H + min(dr, 0), max(dc, 0):W + min(dc, 0)]
    B = b[..., max(-dr, 0):H + min(-dr, 0), max(-dc, 0):W + min(-dc, 0)]
    return float(np.nanmean(np.abs(A - B)))

rows = []
for sh in [(step, 0), (0, step)]:
    v = tangent_tiles_for(kids[few], shift=sh, fill="nan")[0]
    attendu, oppose = mismatch(base, v, *sh), mismatch(base, v, -sh[0], -sh[1])
    rows.append((sh, attendu, oppose))
    sens = "OK" if attendu < oppose else "INVERSÉ"
    print(f"shift={sh}:  écart si sud/est (attendu) {attendu:.4f}   si nord/ouest {oppose:.4f}   -> {sens}")

# visuel : une tuile, base vs fenêtre décalée de (step, 0), sur une bande de 3 patchs
v = tangent_tiles_for(kids[[t]], shift=(step, 0), fill="nan")[0][0]
r0 = H // 2 - 24
fig, axs = plt.subplots(3, 1, figsize=(12, 7))
for a, (img, ttl) in zip(axs, [(base[list(few).index(t)] if t in few else tangent_tiles_for(kids[[t]], fill="nan")[0][0], "base, lignes r0..r0+48"),
                               (v, f"décalée shift=({step},0), mêmes lignes"),
                               (None, "")]):
    if img is None: break
    a.imshow(show(np.nan_to_num(img[:, r0:r0 + 48], nan=float(lo.mean())))); a.set_title(ttl, fontsize=9)
    a.set_xticks([]); a.set_yticks([])
b0 = tangent_tiles_for(kids[[t]], fill="nan")[0][0]
prof = lambda im: np.nanmean(im[:, :, W // 2 - 32:W // 2 + 32], axis=(0, 2))
axs[2].plot(prof(b0), label="base"); axs[2].plot(prof(v), label=f"décalée ({step},0)")
axs[2].plot(np.arange(H) - step, prof(b0), "k:", lw=.8, label="attendu : fenêtre déplacée vers le sud, le contenu remonte")
axs[2].set_xlim(r0, r0 + 48); axs[2].legend(fontsize=8); axs[2].set_xlabel("ligne (pixel)")
axs[2].set_title("profil N→S au centre de la tuile", fontsize=9)
plt.tight_layout(); plt.show()

## Test 2b — `GetDINOV3SAT(over_sample=n)` de bout en bout, sans les poids

On remplace DINOv3 par un backbone factice : chaque token vaut la **couleur moyenne de son patch 16 × 16**. On connaît alors
la réponse exacte. Le token d'une cellule doit valoir la moyenne des données sur la fenêtre de 16 × 16 px, nord en haut,
centrée sur cette cellule. Si l'entrelacement est juste, l'écart doit **diminuer** quand `n` augmente : la fenêtre se
rapproche de la cellule. S'il est faux, `n=2` n'apporte rien par rapport à `n=1`, voire dégrade le résultat.

Pour limiter la mémoire, le test ne prend que le bloc central `CORE × CORE` de tuiles.

In [ ]:
import torch, torch.nn as nn

class MeanDino(nn.Module):
    # « backbone » exact : token = moyenne RGB du patch 16x16, CLS = moyenne des tokens
    def __init__(self):
        super().__init__(); self.dummy = nn.Parameter(torch.zeros(1))
    def forward_features(self, x):
        t = nn.functional.avg_pool2d(x, D.DINOV3_PATCH).flatten(2).transpose(1, 2)
        return {"x_norm_clstoken": t.mean(1), "x_norm_patchtokens": t}

CORE = 4                                              # bloc central CORE x CORE tuiles
core = kids[(np.abs(ROW - (GRID - 1) / 2) < CORE / 2) & (np.abs(COL - (GRID - 1) / 2) < CORE / 2)]
mcore = np.isin(ids >> (2 * k), core)
d_core = np.where(np.isfinite(rgb[mcore]), rgb[mcore], np.nanmean(rgb[mcore], 0))
rng = np.random.default_rng(0)
for n in (1, 2, 4):
    r = D.GetDINOV3SAT(d_core, ids[mcore], LEVEL, PARENT_LEVEL, projection="tangent", over_sample=n,
                       model=MeanDino(), return_patches=True, device="cpu",
                       mean=(0, 0, 0), std=(1, 1, 1))            # pas de normalisation : on compare des réflectances
    pick = rng.choice(r.patch_cell_id.size, min(3000, r.patch_cell_id.size), replace=False)
    lon_c, lat_c = D.cell_centres_lonlat(r.patch_level, r.patch_cell_id[pick])
    glon, glat = D.tangent_grid_lonlat(lon_c, lat_c, D.DINOV3_PATCH, px)
    v, okv = D.sample_healpix(rgb, ids, LEVEL, glon, glat)
    good = okv.reshape(len(pick), -1).all(1)                    # fenêtre entièrement dans les données
    truth = v.reshape(len(pick), -1, 3)[good].mean(1)
    e = r.patch_embedding[pick][good]
    print(f"over_sample={n}  tokens niveau {r.patch_level}  "
          f"|token - fenêtre vraie| = {np.abs(e - truth).mean():.4f}   "
          f"corrélation = {np.mean([np.corrcoef(e[:, c], truth[:, c])[0, 1] for c in range(3)]):.3f}")